<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex10.2-solid-oxide-cell/Ex10.2_04_optimisation_light.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*
*This is the **light** version: every TODO is written out, and you only replace the `...` marked lines with what the comment beside them says.*


<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_10.2 · Notebook 04 — lifetime-aware optimisation

**Paired with L10.2 · Solid oxide cells**

**The course finale.** Everything from L8 to L10 arrives here.

Optimise a 24-hour operating trajectory — current density and temperature —
against a time-varying electricity price, subject to an end-of-life constraint
on the area-specific resistance.

$$\max_{i(t),\,T(t)} \int \left[p_{H_2}\dot n_{H_2}
- c_e\,iAV\right]dt
\qquad \text{s.t.} \quad \mathrm{ASR}(t_{life}) \le \mathrm{ASR}_{max}$$

Nothing here trains a PINN. The decision variables are the trajectory itself,
and what is being differentiated through is the cell model in `problem.py` —
which is why every function in it works on tensors as well as arrays.

---

## 0 · Setup

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex10.2-solid-oxide-cell/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
# --- setup: every Part 2 notebook opens with this cell ------------------
# Needs course_core.py, pinn_core.py and problem.py beside this notebook.
# On Colab the files cell above fetched them from the public course repository.
import os
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"

from pinn_core import *                                  # noqa: F401,F403
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt

set_seed(88)
print("device:", DEVICE, " dtype:", torch.get_default_dtype())

In [ ]:
par = pb.SOCParams("SOEC", 1073.15, 0.10, 0.90)
price = pb.price_profile(24, "daily")
plt.figure(figsize=(7, 2.6)); plt.step(np.arange(24), price, where="mid")
plt.xlabel("hour"); plt.ylabel("electricity price"); plt.tight_layout(); plt.show()

## TODO 1 — a baseline

Run at constant current density and evaluate. This is what a plant without
optimisation does, and it is the number you must beat.

In [ ]:
base = pb.evaluate_trajectory(np.full(24, 1.0), np.full(24, par.T), par, price)
print(f"baseline profit {base['profit']:.2f}   ASR_end {base['ASR_end']:.4f}")
pb.plot_trajectory(base, price)

## TODO 2 — optimise the trajectory

Two routes, and you should do **both**:

**(a) Gradient-based, through a differentiable model.** Represent the
trajectory in torch, evaluate the objective, and let autograd supply
∂J/∂u. This is the point of L10.2 slide 13.

**(b) Brute force on the reference solver.** A coarse grid or random search.
Slow, but it owes nothing to the surrogate.

Then compare. If they agree, you have evidence the differentiable route works.
If they disagree, the optimiser has found an error in your model and exploited
it — L10.2 slide 15, met in person. **Either outcome is a result.**

In [ ]:
# TODO 2a --- gradient-based optimisation through a differentiable model ----------------------------------
# Three `...` to replace:
#   line 1  ->  (H_VALUE * n_H2 - price_t * P_elec).sum()                       profit: hydrogen value minus electricity cost
#   line 2  ->  torch.relu(asr_life - ASR_LIMIT) ** 2                           the lifetime constraint, penalised
#   line 3  ->  torch.optim.Adam([i_t, T_t], lr=1e-2)                           free tensors, not a module
H_VALUE, ASR_LIMIT = 3.0, 0.50
price_t = to_tensor(price.reshape(-1, 1))
E_nernst = float(pb.nernst(par))

def model_profit(i_t, T_t, t_life):
    """The same physics as pb.evaluate_trajectory, written in torch so autograd can differentiate it."""
    asr_T = par.ASR_0 * torch.exp(par.E_act_ASR / pb.R_GAS * (1.0 / T_t - 1.0 / par.T_ref))
    rate  = par.deg_k0 * torch.exp(-par.E_deg / (pb.R_GAS * T_t)) * torch.abs(i_t) ** par.deg_exponent
    asr   = asr_T + torch.cumsum(rate, dim=0)                       # ASR is a state that grows with use
    V = E_nernst + pb.eta_activation(i_t, par) + i_t * asr + torch.sign(i_t) * pb.eta_concentration(i_t, par)
    n_H2   = i_t * (par.area * 1e4) / (2.0 * pb.F_CONST) * 3600
    P_elec = i_t * (par.area * 1e4) * V / 1000.0
    profit = ...                                  # <- (H_VALUE * n_H2 - price_t * P_elec).sum()
    asr_life = par.ASR_0 + (asr[-1] - par.ASR_0) * t_life / 24.0    # today's degradation, repeated over the required life
    return profit, asr_life

def optimise(t_life=20_000, steps=800, w=1e4):
    i_t = to_tensor(np.full(24, 1.0).reshape(-1, 1), requires_grad=True)
    T_t = to_tensor(np.full(24, par.T).reshape(-1, 1), requires_grad=True)
    opt = ...                                     # <- torch.optim.Adam([i_t, T_t], lr=1e-2)
    hist = []
    for step in range(steps):
        opt.zero_grad()
        profit, asr_life = model_profit(i_t, T_t, t_life)
        J = -profit + w * ...                     # <- torch.relu(asr_life - ASR_LIMIT) ** 2
        J.backward(); opt.step(); hist.append(float(J))
        with torch.no_grad():
            i_t.clamp_(0.0, 0.9 * par.i_L); T_t.clamp_(923.15, 1173.15)   # stay inside the model's validity
    return to_numpy(i_t).ravel(), to_numpy(T_t).ravel(), hist

i_opt, T_opt, hist = optimise()
plot_curves({"adam": np.asarray(hist)}, title="J = -profit + penalty"); plt.show()
ref = pb.evaluate_trajectory(i_opt, T_opt, par, price)
print(f"gradient route: profit {ref['profit']:.2f}   ASR_end {ref['ASR_end']:.4f}   (baseline {base['profit']:.2f})")
pb.plot_trajectory(ref, price)
# ------------------------------------------------------------------------------

In [ ]:
# TODO 2b --- brute force on the reference solver ---------------------------------------------------------------
# Two `...` to replace:
#   line 1  ->  pb.evaluate_trajectory(np.full(24, i_c), np.full(24, T_c), par, price)
#   line 2  ->  par.ASR_0 + (r["ASR_end"] - par.ASR_0) * 20_000 / 24.0 <= ASR_LIMIT      the same lifetime rule
best = None
for i_c in np.linspace(0.4, 1.8, 15):
    for T_c in (973.15, 1023.15, 1073.15, 1123.15, 1173.15):
        r = ...                                   # <- pb.evaluate_trajectory(np.full(24, i_c), np.full(24, T_c), par, price)
        feasible = ...                            # <- par.ASR_0 + (r["ASR_end"] - par.ASR_0) * 20_000 / 24.0 <= ASR_LIMIT
        if feasible and (best is None or r["profit"] > best["profit"]):
            best = r
print(f"brute force : profit {best['profit']:.2f}   i {best['i'][0]:.2f} A/cm2   T {best['T'][0]-273.15:.0f} C")
print(f"gradient    : profit {ref['profit']:.2f}   mean i {i_opt.mean():.2f}   mean T {T_opt.mean()-273.15:.0f} C")
print("Agreement is evidence the differentiable route works; disagreement means the optimiser found a")
print("hole in the torch model and exploited it. Say which you trust, and why, in the report.")
# ------------------------------------------------------------------------------

## TODO 3 — vary the required lifetime

Re-optimise for a required lifetime of 5 000, 20 000 and 40 000 hours. Watch
the optimal trajectory move.

This is the clearest demonstration in the whole course that temperature is a
**trade-off and not a setting**: the same cell, the same prices, and a
different answer purely because the device must last longer.

In [ ]:
# TODO 3 --- vary the required lifetime -----------------------------------------------------------------------
# One `...` to replace:  optimise(t_life=t_life)
fig, axes = plt.subplots(1, 2, figsize=(12.0, 3.8))
for t_life in (5_000, 20_000, 40_000):
    i_l, T_l, _ = ...                             # <- optimise(t_life=t_life)
    r = pb.evaluate_trajectory(i_l, T_l, par, price)
    asr_life = par.ASR_0 + (r["ASR_end"] - par.ASR_0) * t_life / 24.0
    binding = "binding" if asr_life > 0.95 * ASR_LIMIT else "NOT binding"
    print(f"life {t_life:>6,} h: profit {r['profit']:.2f}   ASR at end of life {asr_life:.3f}   ({binding})")
    axes[0].step(range(24), i_l, where="mid", label=f"{t_life:,} h")
    axes[1].step(range(24), T_l - 273.15, where="mid", label=f"{t_life:,} h")
axes[0].set_ylabel("i [A/cm2]"); axes[1].set_ylabel("T [degC]")
for a in axes: a.set_xlabel("hour"); a.legend(frameon=False, fontsize=9); a.grid(alpha=0.25)
plt.tight_layout(); plt.show()
# ------------------------------------------------------------------------------

## Save

Notebook 05 reads every case it finds in `Ex10.2_outputs` and puts one row in
the report per case.

In [ ]:
import pickle

os.makedirs("Ex10.2_outputs", exist_ok=True)
# with open(os.path.join("Ex10.2_outputs", "ex102_opt.pkl"), "wb") as f:
#     pickle.dump([{"par": par, "OCV": float(pb.nernst(par)),
#                   "i_tn": ..., "profit": ..., "ASR_end": ...}], f)